In [ ]:
import os
from flask import Flask, render_template_string, request
from google.cloud import storage

app = Flask(__name__)


GCS_BUCKET_NAME = "mybucketprojectvipul"


UPLOAD_FOLDER = "uploads"
os.makedirs(UPLOAD_FOLDER, exist_ok=True)
app.config["UPLOAD_FOLDER"] = UPLOAD_FOLDER


os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = '/Users/vipulsingh/Desktop/untitled folder/DataEngineering/my-project-vipulsingh-e76a3081429d.json'

HTML_CONTENT = """
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Walmart Stores in USA - File Upload</title>
    <style>
        body {
            font-family: 'Arial', sans-serif;
            background-color: #f7f7f7;
            margin: 0;
            padding: 0;
            color: #333;
        }
        header {
            background-color: #2d87f0;
            color: white;
            padding: 20px 0;
            text-align: center;
        }
        header h1 {
            font-size: 36px;
            margin: 0;
            text-transform: uppercase;
        }
        .container {
            width: 80%;
            margin: 30px auto;
            background-color: white;
            padding: 20px;
            box-shadow: 0 4px 8px rgba(0, 0, 0, 0.1);
            border-radius: 8px;
        }
        .form-container {
            text-align: center;
            padding: 20px;
        }
        .form-container input[type="file"] {
            font-size: 16px;
            padding: 10px;
            margin: 10px 0;
        }
        .form-container button {
            background-color: #2d87f0;
            color: white;
            font-size: 16px;
            padding: 10px 20px;
            border: none;
            border-radius: 4px;
            cursor: pointer;
        }
        .form-container button:hover {
            background-color: #1c65b4;
        }
        footer {
            background-color: #2d87f0;
            color: white;
            text-align: center;
            padding: 10px;
            position: fixed;
            bottom: 0;
            width: 100%;
        }
    </style>
</head>
<body>
    <header>
        <h1>Walmart Stores in United States of America</h1>
    </header>

    <div class="container">
        <div class="form-container">
            <h2>Upload a File to Google Cloud Storage</h2>
            <form action="/upload" method="POST" enctype="multipart/form-data">
                <label for="file">Choose a file to upload:</label><br>
                <input type="file" name="file" id="file" required><br>
                <button type="submit">Upload File</button>
            </form>
        </div>
    </div>

    <footer>
        <p>&copy; 2025 Walmart Stores</p>
    </footer>
</body>
</html>
"""

@app.route("/")
def index():
    return render_template_string(HTML_CONTENT)

@app.route("/upload", methods=["POST"])
def upload_file():
    if "file" not in request.files:
        return "No file part in the request", 400
    
    file = request.files["file"]
    
    if file.filename == "":
        return "No file selected", 400

    print(f"Received file: {file.filename}")  

    if file:
        
        local_path = os.path.join(app.config["UPLOAD_FOLDER"], file.filename)
        file.save(local_path)

        
        print(f"File saved locally at: {local_path}")

        
        try:
            upload_to_gcs(local_path, file.filename)
            print(f"File '{file.filename}' uploaded successfully to GCS!") 
            return f"File '{file.filename}' uploaded successfully to GCS!", 200
        except Exception as e:
            print(f"Error uploading file to GCS: {str(e)}")  
            return f"Error uploading file to GCS: {str(e)}", 500
        finally:
            
            os.remove(local_path)

def upload_to_gcs(file_path, file_name):
    """Uploads a file to Google Cloud Storage."""
    
    storage_client = storage.Client(project='my-project-vipulsingh')
    bucket = storage_client.bucket(GCS_BUCKET_NAME)
    blob = bucket.blob(file_name)
    
    try:
        blob.upload_from_filename(file_path)
        print(f"File uploaded to GCS: {file_name}")
    except Exception as e:
        print(f"Error uploading file to GCS: {str(e)}")
        raise e

if __name__ == "__main__":
    app.run(debug=True)
